# EDA: Calima vs Deaths — Fuerteventura

**Objective:** Analyze the association between calima (proxy) and weekly mortality in Fuerteventura, including lagged effects (lag0, lag1, lag2).

**Key variables:**
- `deaths_week`: weekly deaths (2016–2025)
- `calima_proxy_score`: heuristic index (0–4.5)
- `calima_proxy_level`: category (no_calima / possible / probable / intense)

**Sections:**
1. Load data
2. Lag0, lag1, lag2 correlations
3. Group by proxy category
4. Significant differences (ANOVA) and effect sizes (Δ deaths)
4.1 Pairwise comparisons
5. Visualizations
6. Summary

---

## 1. Load Data

In [7]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# ─── ISLAND CONFIG ─────────────────────────────────────────────────────────────
ISLAND_NAME = "fuerteventura"   # e.g. "gran_canaria", "tenerife", "lanzarote"
ISLAND_CODE = "ftv"   # e.g. "gcan", "tfe", "lanz"
# ───────────────────────────────────────────────────────────────────────────────

CWD = Path.cwd().resolve()

# If running from islands/<island>, go up two levels to repo root
if CWD.name == ISLAND_NAME and CWD.parent.name == "islands":
    ROOT = CWD.parent.parent
else:
    ROOT = CWD

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("CWD :", CWD)
print("ROOT:", ROOT)
print("src exists?:", (ROOT / "src").exists())

from src.utils.d25_nb_utils import (
    section, glance, checks, missing_table, num_summary,
    autosave_fig, save_table,
)

# Output directories
REPORTS_DIR = ROOT / "reports" / "islands"
FIG_DIR = REPORTS_DIR / "figures" / ISLAND_NAME
TAB_DIR = REPORTS_DIR / "tables" / ISLAND_NAME

FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

print("FIG_DIR:", FIG_DIR)
print("TAB_DIR:", TAB_DIR)

# Load master dataset
FP = ROOT / "data/processed" / ISLAND_NAME / "master" / f"master_{ISLAND_CODE}_2016_2025.parquet"
print("FP:", FP)
assert FP.exists(), f"Missing file: {FP}"

section(f"EDA core weekly {ISLAND_NAME}")

df = pd.read_parquet(FP)
print("Loaded:", FP)

df["week_start"] = pd.to_datetime(df["week_start"], errors="coerce")
print("Week range:", df["week_start"].min(), "->", df["week_start"].max())
glance(df, label=f"eda_core_weekly_{ISLAND_CODE}", n=5)

checks(
    df,
    required=["week_start", "deaths_week"],
    key=["week_start"],
    dt="week_start"
)

num_summary(df)

# Load calima proxy dataset
calima_fp = ROOT / "data" / "processed" / ISLAND_NAME / "calima" / f"calima_proxy_v2_weekly_{ISLAND_CODE}_2016_2025.parquet"
print("Calima proxy FP:", calima_fp)

if calima_fp.exists():
    calima = pd.read_parquet(calima_fp).copy()
    calima["week_start"] = pd.to_datetime(calima["week_start"], errors="coerce")
    keep = [
        "week_start",
        "calima_proxy_score",
        "calima_proxy_level",
    ]

    calima_keep = [c for c in keep if c in calima.columns]

    # Drop overlapping columns before merge to avoid duplicates
    overlap = [c for c in calima_keep if c != "week_start" and c in df.columns]
    if overlap:
        print("Dropping overlapping columns before merge:", overlap)
        df = df.drop(columns=overlap)

    df = df.merge(calima[calima_keep], on="week_start", how="left")
    print("Merged calima proxy columns:", [c for c in calima_keep if c != "week_start"])
else:
    print("Calima proxy weekly dataset not found. Section 6.1 will be skipped.")

print("Final shape:", df.shape)

CWD : C:\Users\fdora\RA_Career\Projects\climate_mortality\islands\fuerteventura
ROOT: C:\Users\fdora\RA_Career\Projects\climate_mortality
src exists?: True
FIG_DIR: C:\Users\fdora\RA_Career\Projects\climate_mortality\reports\islands\figures\fuerteventura
TAB_DIR: C:\Users\fdora\RA_Career\Projects\climate_mortality\reports\islands\tables\fuerteventura
FP: C:\Users\fdora\RA_Career\Projects\climate_mortality\data\processed\fuerteventura\master\master_ftv_2016_2025.parquet

EDA core weekly fuerteventura
Loaded: C:\Users\fdora\RA_Career\Projects\climate_mortality\data\processed\fuerteventura\master\master_ftv_2016_2025.parquet
Week range: 2015-12-28 00:00:00 -> 2025-12-29 00:00:00

--- eda_core_weekly_ftv ---
shape: (523, 42)

dtypes:
 week_start                     datetime64[ns]
year                                    int32
island                                 object
island_code                            object
deaths_week                           float64
deaths_missing_week          

,week_start,year,island,island_code,deaths_week,deaths_missing_week,n_days,temp_c_mean,tmax_c_mean,tmax_c_max,...,O3,days_with_pm10,days_missing_pm10,cap_heat_level_max_week,cap_dust_level_max_week,cap_heat_yellow_plus_week,cap_dust_yellow_plus_week,cap_coverage_week,calima_dai_flag,calima_level_week
0,2015-12-28,2015,fuerteventura,ftv,NaN,NaN,3,19.366667,21.766667,21.8,...,79.333333,3,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0
1,2016-01-04,2016,fuerteventura,ftv,3.0,0.0,7,19.014286,22.442857,24.1,...,78.142857,7,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0
2,2016-01-11,2016,fuerteventura,ftv,9.0,0.0,7,19.171429,22.285714,23.0,...,76.571429,7,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0
3,2016-01-18,2016,fuerteventura,ftv,5.0,0.0,7,19.514286,22.428571,23.2,...,79.142857,7,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0
4,2016-01-25,2016,fuerteventura,ftv,7.0,0.0,7,19.771429,22.342857,23.2,...,89.142857,7,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0


Calima proxy FP: C:\Users\fdora\RA_Career\Projects\climate_mortality\data\processed\fuerteventura\calima\calima_proxy_v2_weekly_ftv_2016_2025.parquet
Merged calima proxy columns: ['calima_proxy_score', 'calima_proxy_level']
Final shape: (523, 44)


## 2. Lags

In [8]:


print(f"Rows after filtering first week: {len(df)}")
print(f"Deaths nulls: {df['deaths_week'].isna().sum()}")
print(f"Calima proxy score nulls: {df['calima_proxy_score'].isna().sum()}")

# Create lag variables for calima_proxy_score
df['calima_proxy_score_lag1'] = df['calima_proxy_score'].shift(1)
df['calima_proxy_score_lag2'] = df['calima_proxy_score'].shift(2)

print("\n✅ Lag variables created:")
print(f"  lag0 (contemporaneous): {df['calima_proxy_score'].notna().sum()} non-null")
print(f"  lag1 (1 week prior):    {df['calima_proxy_score_lag1'].notna().sum()} non-null")
print(f"  lag2 (2 weeks prior):   {df['calima_proxy_score_lag2'].notna().sum()} non-null")

Rows after filtering first week: 523
Deaths nulls: 1
Calima proxy score nulls: 0

✅ Lag variables created:
  lag0 (contemporaneous): 523 non-null
  lag1 (1 week prior):    522 non-null
  lag2 (2 weeks prior):   521 non-null


In [9]:
# Correlations: deaths_week vs calima_proxy_score at different lags
corr_lag0 = df['deaths_week'].corr(df['calima_proxy_score'])
corr_lag1 = df['deaths_week'].corr(df['calima_proxy_score_lag1'])
corr_lag2 = df['deaths_week'].corr(df['calima_proxy_score_lag2'])

corr_summary = pd.DataFrame({
    'lag': ['lag0 (same week)', 'lag1 (1 week prior)', 'lag2 (2 weeks prior)'],
    'correlation': [corr_lag0, corr_lag1, corr_lag2],
    'n_pairs': [
        df[['deaths_week', 'calima_proxy_score']].notna().all(axis=1).sum(),
        df[['deaths_week', 'calima_proxy_score_lag1']].notna().all(axis=1).sum(),
        df[['deaths_week', 'calima_proxy_score_lag2']].notna().all(axis=1).sum(),
    ]
})

print("Correlations: deaths_week vs calima_proxy_score\n")
print(corr_summary.to_string(index=False))

# Save
corr_summary.to_csv(TAB_DIR / 'calima_deaths_lags_correlation.csv', index=False)
print("\n✅ Saved: calima_deaths_lags_correlation.csv")

Correlations: deaths_week vs calima_proxy_score

                 lag  correlation  n_pairs
    lag0 (same week)     0.011470      522
 lag1 (1 week prior)    -0.001674      522
lag2 (2 weeks prior)     0.072412      521

✅ Saved: calima_deaths_lags_correlation.csv


**Lag Analysis — Fuerteventura:**

| Lag | Correlation | Interpretation |
|-----|-------------|-----------------|
| **lag0 (same week)** | 0.011 | Strongest but near-zero |
| lag1 (1 week prior) | -0.002 | Essentially zero |
| **lag2 (2 weeks prior)** | 0.072 | Slightly stronger but still weak |

**Interpretation:**

✓ **Strongest at lag0 (r=0.011)** technically, but **all correlations essentially zero**.

**Key findings:**

1. **No meaningful correlation:**
   - lag0 = 0.011 is **noise** (r ≈ 0)
   - lag2 = 0.072 is slightly higher but still very weak
   - Pattern inconsistent (no clear lag structure)

2. **Weak signal at lag2 (unusual):**
   - Only lag2 shows marginally stronger correlation (0.072)
   - Unlike other islands where lag0 dominant
   - Suggests possible **delayed effect** but magnitude negligible

3. **Population context:**
   - Fuerteventura: ~119k population
   - Similar to Lanzarote (149k) where we SAW effect (lag0=0.120)
   - Yet Fuerteventura shows NO signal → unexpected

**Comparison across all islands:**

| Island | lag0 | lag2 | Pattern | Population |
|--------|------|------|---------|-----------|
| Tenerife | 0.221 | 0.191 | Strong lag0 | 928k |
| Gran Canaria | 0.210 | 0.179 | Strong lag0 | 846k |
| Lanzarote | 0.120 | 0.059 | Weak lag0 | 149k |
| Fuerteventura | 0.011 | 0.072 | No signal | 119k |
| La Palma | 0.020 | -0.037 | No signal | 83k |
| Gomera | -0.007 | -0.031 | No signal | 23k |

**Key anomaly:** Fuerteventura similar size to Lanzarote but **no detectable lag0 effect** (0.011 vs 0.120).

---

## 3. Group by Proxy Category

In [10]:
# Group by calima_proxy_level and compute deaths statistics
level_order = ['no_calima', 'possible', 'probable', 'intense']

deaths_by_level = (
    df.groupby('calima_proxy_level', observed=True)['deaths_week']
    .agg(['count', 'mean', 'median', 'std', 'min', 'max'])
    .reindex(level_order)
)

print("Deaths statistics by calima proxy level:\n")
print(deaths_by_level.round(2))

# Compute Δ deaths (intense vs baseline)
baseline = deaths_by_level.loc['no_calima', 'mean']
intense  = deaths_by_level.loc['intense', 'mean']
delta    = intense - baseline

print(f"\nΔ deaths (intense vs no_calima): {delta:.2f} deaths/week")

# Save
deaths_by_level.to_csv(TAB_DIR / 'calima_level_v_deaths_stats.csv')
print("\n✅ Saved: calima_level_v_deaths_stats.csv")

Deaths statistics by calima proxy level:

                    count   mean  median   std  min   max
calima_proxy_level                                       
no_calima             315   9.87    10.0  3.45  2.0  22.0
possible               99  10.07    10.0  3.49  3.0  20.0
probable               70   9.77     9.0  3.35  4.0  18.0
intense                38  10.11    10.0  3.23  3.0  17.0

Δ deaths (intense vs no_calima): 0.23 deaths/week

✅ Saved: calima_level_v_deaths_stats.csv


**Interpretation: Deaths by Calima Proxy Level — Fuerteventura:**

**Finding:** No dose-response; essentially flat across all levels.

| Level | Mean deaths/week | Δ vs baseline | % increase |
|-------|-----------------|---------------|-----------|
| **no_calima** | 9.87 | — | — |
| **possible** | 10.07 | +0.20 | +2.0% |
| **probable** | 9.77 | -0.10 | -1.0% |
| **intense** | 10.11 | **+0.23** | **+2.3%** |

**Key findings:**

1. **No monotonic gradient:**
   - no_calima → possible: +2.0%
   - no_calima → probable: -1.0% (inverse)
   - no_calima → intense: +2.3%
   - Pattern is **flat/erratic, not dose-dependent**

2. **Minimal intense effect:**
   - Only +0.23 deaths/week (smallest of all islands)
   - Smallest % increase (2.3%)
   - Comparable to La Palma (1.12 deaths/week) despite similar population to Lanzarote

3. **High baseline variability:**
   - All categories have std ~3.3-3.5
   - Weekly deaths ~10/week → variability ~35% of mean
   - Very high noise floor

4. **Comparison with Lanzarote (similar population):**

| Island | Population | lag0 | Δ deaths | Pattern |
|--------|-----------|------|----------|---------|
| **Lanzarote** | 149k | 0.120 | 2.16 | Weak positive |
| **Fuerteventura** | 119k | 0.011 | 0.23 | Flat |

**Epidemiological interpretation:**
- Fuerteventura smaller than Lanzarote → even weaker signal
- No dose-response suggests **calima not detectable** in mortality
- Effect size negligible compared to baseline noise

**Conclusion:** Fuerteventura shows **no meaningful calima-mortality association**. Effect flat across all intensity levels with minimal excess at intense (+0.23 deaths/week).

---
======================================================================
Fuerteventura: Calima-Mortality Analysis — Insufficient Signal
======================================================================

**Finding:** No detectable calima-mortality association in Fuerteventura.

**Evidence:**
- Lag0 correlation: r = 0.011 (essentially zero)
- All lags near-zero or inconsistent (lag2 = 0.072, still weak)
- Deaths by proxy level: flat across all intensities (Δ = +0.23 deaths/week)
- Population: ~119,000 (smaller than Lanzarote, yet weaker signal)
- Weekly deaths: ~10/week (high noise-to-signal ratio)

**Comparison:** Despite similar population to Lanzarote (149k, which showed weak but detectable effect: lag0=0.120, Δ=2.16), 
Fuerteventura shows negligible correlation and minimal excess mortality.

**Conclusion:** Fuerteventura's calima-mortality association undetectable. 
Weekly mortality variability dominates. Population size and/or local epidemiology insufficient for signal detection.

**Analysis discontinued for Fuerteventura.**